In [4]:
import nltk
import pandas as pd
import ast

df = pd.read_csv('/Users/keremozemre/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/social compute/Social_computational_science_Project/politicians_network_nodes_with_selected_text.csv')

In [14]:
df["positions_text"].isna().sum()

np.int64(3277)

In [15]:
df["combined_text"] = (
    df["intro_text"].fillna("") + 
    df["career_text"].fillna("") + 
    df["positions_text"].fillna("") + 
    df["article_text"].fillna("")
)

In [16]:
df["combined_text"].head(10)

0    Donald John Trump (born June 14, 1946) is an A...
1    James Earl Carter Jr. (October 1, 1924 – Decem...
2    Karl Christian Rove (born December 25, 1950) i...
3    William Jefferson  Clinton (born William Jeffe...
4    Douglas Brian Sosnik (born September 26, 1956)...
5    Ronald Wilson Reagan (February 6, 1911 – June ...
6    Rahm Israel Emanuel (; born November 29, 1959)...
7    Richard Milhous Nixon (January 9, 1913 – April...
8    John David Podesta Jr. (born January 8, 1949) ...
9    Anita Dunn (née Babbitt; born January 8, 1958)...
Name: combined_text, dtype: object

In [17]:
import nltk
import re
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def tokenize(text):
    text = text.lower()
    
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    stems = [stemmer.stem(word) for word in filtered_words]
    
    
    return stems

df["tokens"] = df["combined_text"].dropna().apply(tokenize)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/keremozemre/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [18]:
df["tokens"]

0       [donald, john, trump, born, june, american, po...
1       [jame, earl, carter, jr, octob, decemb, americ...
2       [karl, christian, rove, born, decemb, american...
3       [william, jefferson, clinton, born, william, j...
4       [dougla, brian, sosnik, born, septemb, america...
                              ...                        
4102    [alveda, celest, king, born, januari, american...
4103    [richard, allan, hill, born, decemb, american,...
4104    [jame, wadsworth, symington, symingtn, born, s...
4105    [marvin, henri, mickey, edward, born, juli, am...
4106    [richard, walker, mallari, februari, septemb, ...
Name: tokens, Length: 4107, dtype: object

In [19]:
import re
import math
import numpy as np
from collections import Counter
from scipy.stats import chi2
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
import nltk

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'R': wordnet.ADV}
    return tag_map.get(tag, wordnet.NOUN)

def tokenize_lemmatized(abstract):
    """Clean → remove stopwords → lemmatize. Used for bigram discovery."""
    if not isinstance(abstract, str):
        return []
    text = abstract.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [w for w in text.split() if w not in stop_words]
    return [lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in words]

df["tokens_lemmatized"] = df["combined_text"].apply(tokenize_lemmatized)
df[["tokens", "tokens_lemmatized"]].head()

,tokens,tokens_lemmatized
0,"[donald, john, trump, born, june, american, po...","[donald, john, trump, born, june, american, po..."
1,"[jame, earl, carter, jr, octob, decemb, americ...","[james, earl, carter, jr, october, december, a..."
2,"[karl, christian, rove, born, decemb, american...","[karl, christian, rove, born, december, americ..."
3,"[william, jefferson, clinton, born, william, j...","[william, jefferson, clinton, born, william, j..."
4,"[dougla, brian, sosnik, born, septemb, america...","[douglas, brian, sosnik, born, september, amer..."


In [20]:
from nltk.probability import FreqDist

all_tokens = df["tokens_lemmatized"].explode().tolist()

fq = FreqDist(all_tokens)
fq.most_common(10)

[('serve', 19389),
 ('state', 18551),
 ('republican', 11142),
 ('house', 10125),
 ('district', 9732),
 ('american', 9411),
 ('representative', 9108),
 ('u', 8980),
 ('member', 8973),
 ('united', 8842)]

In [21]:
clean_tokens = [t for t in all_tokens if t is not None and not (isinstance(t, float) and math.isnan(t))]

bigrams = list(nltk.bigrams(clean_tokens))

bigram_counts = Counter(bigrams)
w1_counts     = Counter([w1 for w1, _ in bigrams])
w2_counts     = Counter([w2 for _, w2 in bigrams])
N             = len(bigrams)

chi2_scores  = {}
p_values     = {}
co_occurences = {}

for (w1, w2), nii in bigram_counts.items():
    nio = w1_counts[w1] - nii
    noi = w2_counts[w2] - nii
    noo = N - (nii + nio + noi)

    observed = np.array([[nii, nio], [noi, noo]])
    row_sums  = observed.sum(axis=1)
    col_sums  = observed.sum(axis=0)
    expected  = np.outer(row_sums, col_sums) / N

    chi2_stat = ((observed - expected) ** 2 / expected).sum()
    p_val     = chi2.sf(chi2_stat, df=1)

    co_occurences[(w1, w2)] = nii
    chi2_scores[(w1, w2)]   = chi2_stat
    p_values[(w1, w2)]      = p_val

print(f"Total bigram types: {len(bigram_counts):,}")

Total bigram types: 276,256


In [22]:
filtered_bigrams = {
    bg:(co_occurences[bg]) for bg in bigram_counts
    if (p_values[bg] < 0.001) and (co_occurences[bg] > 50)
}

In [23]:
sorted_bigrams = sorted(
    filtered_bigrams.items(),
    key=lambda x: x[1],
    reverse=True
)
print(sorted_bigrams)

[(('united', 'state'), 8484), (('house', 'representative'), 5617), (('american', 'politician'), 3983), (('congressional', 'district'), 3618), (('politician', 'serve'), 2608), (('democratic', 'party'), 2482), (('state', 'senate'), 2446), (('state', 'house'), 2386), (('republican', 'party'), 2249), (('new', 'york'), 2196), (('th', 'congressional'), 2162), (('u', 'representative'), 2144), (('member', 'democratic'), 1600), (('serve', 'u'), 1496), (('th', 'district'), 1474), (('member', 'republican'), 1419), (('u', 'house'), 1377), (('u', 'senate'), 1339), (('attorney', 'general'), 1321), (('serve', 'united'), 1296), (('general', 'election'), 1279), (('lieutenant', 'governor'), 1252), (('state', 'senator'), 1237), (('serve', 'th'), 1206), (('serve', 'member'), 936), (('member', 'united'), 926), (('previously', 'serve'), 848), (('american', 'lawyer'), 826), (('u', 'senator'), 813), (('new', 'jersey'), 804), (('law', 'school'), 774), (('democratic', 'primary'), 771), (('city', 'council'), 749

In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import MWETokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()


def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'R': wordnet.ADV}
    return tag_map.get(tag, wordnet.NOUN)  # default to NOUN

collocations  = list(filtered_bigrams.keys())
mwe_tokenizer = MWETokenizer(collocations, separator='_')

def tokenize(abstract):
    """
    Full pipeline:
      1. Lowercase & clean
      2. Remove stopwords
      3. Merge known collocations (MWE)
      4. Lemmatize single-word tokens (collocations left as-is)
    """
    if not isinstance(abstract, str):
        return []
    text = abstract.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = [w for w in text.split() if w not in stop_words]

    # Merge collocations first
    tokens = mwe_tokenizer.tokenize(words)

    # Lemmatize only single-word tokens; leave collocation tokens (contain '_') as-is
    lemmatized = [
        token if '_' in token else lemmatizer.lemmatize(token, get_wordnet_pos(token))
        for token in tokens
    ]
    return lemmatized

df["tokens"] = df["combined_text"].apply(tokenize)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/keremozemre/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/keremozemre/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


0    [donald, john, trump, born_june, american_poli...
1    [james, earl, carter, jr, october, december_am...
2    [karl, christian, rove, born_december, america...
3    [william, jefferson, clinton, born, william, j...
4    [douglas, brian, sosnik, born_september, ameri...
Name: tokens, dtype: object

In [26]:
df["tokens"].head(5)

0    [donald, john, trump, born_june, american_poli...
1    [james, earl, carter, jr, october, december_am...
2    [karl, christian, rove, born_december, america...
3    [william, jefferson, clinton, born, william, j...
4    [douglas, brian, sosnik, born_september, ameri...
Name: tokens, dtype: object

In [27]:
df.to_csv("tokenized_wikipage.csv",index = False)